# Tutorial 1: DEG evaluation metrics

This notebook shows how to use the evaluation functions in `script/culc_DEG_metrics.py` on a toy matrix.

1. `DEG_ranking`: Mann-Whitney U statistic of every gene (one cluster vs. the other clusters)
2. `DEG_metric_func`: compares the predicted and the ground-truth U-scores (DEG Spearman and nDCG@k)

Everything runs on the CPU with toy data; no dataset is needed.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # torchmetrics notice
sys.path.insert(0, "../script")   # make script/ importable

import numpy as np
import torch

from culc_DEG_metrics import DEG_ranking, DEG_metric_func

rng = np.random.default_rng(0)
_ = torch.manual_seed(0)

## 1. Toy data

`X_gt` and `X_pred` are the ground-truth and the predicted spot x gene expression matrices, and `cluster_id` is the cluster (e.g. pathologist annotation) of every spot.
Cluster `c` over-expresses the 10 genes `10*c ... 10*c+9` in the ground truth, so these are its true DEGs.
The prediction is the ground truth plus Gaussian noise.

In [2]:
N, G, n_clusters = 300, 100, 3
cluster_id = rng.integers(0, n_clusters, N)

# ground truth: marker genes of cluster c are over-expressed in the spots of cluster c
X_gt = rng.normal(size=(N, G))
for c in range(n_clusters):
    X_gt[np.ix_(cluster_id == c, np.arange(10 * c, 10 * c + 10))] += 2.0
X_gt = torch.tensor(X_gt, dtype=torch.float32)

# prediction: ground truth + noise
X_pred = X_gt + 3.0 * torch.randn_like(X_gt)

print(X_gt.shape, "| spots per cluster:", np.bincount(cluster_id))

torch.Size([300, 100]) | spots per cluster: [ 90  95 115]


## 2. `DEG_ranking`: U-score of every gene

For each cluster, the one-sided Mann-Whitney U test ("the cluster is higher than the rest") is computed for every gene.

* `U_dict[cid]`: U statistic of every gene `[G]` (the larger, the more specific to the cluster; gene order is unchanged)
* `rank_dict[cid]`: gene indices sorted by descending U (the first one is the most specific gene)

In [3]:
rank_gt, U_gt = DEG_ranking(X_gt, cluster_id, device="cpu")       # device="cuda" is used in training
rank_pred, U_pred = DEG_ranking(X_pred, cluster_id, device="cpu")

for cid in rank_gt:
    print("cluster", cid, "| U shape:", tuple(U_gt[cid].shape),
          "| top-10 genes (GT):", sorted(rank_gt[cid][:10].tolist()),
          "| top-10 genes (pred):", sorted(rank_pred[cid][:10].tolist()))

cluster 0 | U shape: (100,) | top-10 genes (GT): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | top-10 genes (pred): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
cluster 1 | U shape: (100,) | top-10 genes (GT): [10, 11, 12, 13, 14, 15, 16, 17, 18, 19] | top-10 genes (pred): [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
cluster 2 | U shape: (100,) | top-10 genes (GT): [20, 21, 22, 23, 24, 25, 26, 27, 28, 29] | top-10 genes (pred): [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


## 3. `DEG_metric_func`: compare predicted and ground-truth U-scores

Both inputs are dicts `{cluster id: U-score [G]}`. It returns

* `spearman`: Spearman correlation between the two U-score vectors, averaged over clusters
* `ndcgBinary@k`: the top-k genes of the ground-truth U-score are treated as relevant, and nDCG@k of the predicted U-score is computed, averaged over clusters

If the prediction were identical to the ground truth, every metric would be 1.0.

In [4]:
res = DEG_metric_func(pred_Uscore=U_pred, gt_Uscore=U_gt, topk_list=[10, 20, 50])
print({k: round(float(v), 3) for k, v in res.items()})

{'spearman': 0.749, 'ndcgBinary@10': 1.0, 'ndcgBinary@20': 0.754, 'ndcgBinary@50': 0.785}


## 4. Notes on how the metrics are used in the paper code

* `topk_list` is free: `main.py --ndcg_topk 50 100 200` gives the nDCG@50/100/200 reported in the paper (the largest k selects the best model).
* The scores are computed **per slide** and averaged over slides (see `evaluate` and `eval_net` in `script/eval.py`).
* At test time, `eval_net` additionally removes the undetermined cluster and clusters with fewer than 10 spots before calling `DEG_metric_func`.
* In training and evaluation the ground-truth U-score comes from `DEG_ranking` on the measured expression of the slide, with the pathologist annotation (test) or k-means clusters (train).
* `DEG_ranking` takes a `gene_chunk` argument (50 by default) which only controls the GPU memory usage, not the result.